# MW Craft 코랩 학습 노트북

이 노트북은 `scripts/colab-train.js`를 호출해서 **로컬 서버 없이 RL 가중치만 학습**하는 용도입니다.

셀을 위에서 아래 순서대로 실행하면 됩니다.

## 이 노트북이 하는 일
- Google Drive 마운트 선택
- Node.js / npm 준비
- 프로젝트 내려받기 또는 이미 올려둔 프로젝트 사용
- `solo` / `selfplay` 학습 실행
- 결과 가중치를 Drive에 저장하거나 바로 다운로드


## 먼저 알아둘 점

1. `REPO_URL`을 넣으면 코랩에서 바로 `git clone` 합니다.
2. `REPO_URL`을 비워두면, 이미 `/content/mwcraft` 같은 경로에 프로젝트를 업로드해둔 상태라고 가정합니다.
3. 결과 가중치를 오래 보관하려면 `MOUNT_DRIVE = True`로 두고 `OUTPUT_DIR`를 Drive 경로로 지정하는 게 안전합니다.


In [ ]:
# ==============================
# 1) 사용자 설정
# ==============================

# GitHub 저장소 주소를 넣으면 코랩에서 바로 clone 합니다.
# 예: REPO_URL = "https://github.com/yourname/mwcraft.git"
# 이미 프로젝트 폴더를 업로드해뒀다면 빈 문자열로 두세요.
REPO_URL = ""

# 코랩 안에서 프로젝트가 위치할 경로입니다.
PROJECT_DIR = "/content/mwcraft"

# 학습 결과를 Google Drive에 저장하고 싶으면 True로 두세요.
MOUNT_DRIVE = True

# 가중치를 복사할 최종 폴더입니다. 빈 문자열이면 복사하지 않습니다.
OUTPUT_DIR = "/content/drive/MyDrive/mwcraft-weights"

# 학습 모드: "solo" 또는 "selfplay"
MODE = "selfplay"

# 난이도: "hard" 또는 "expert"
DIFFICULTY = "expert"

# solo에서는 에피소드 수, selfplay에서는 매치 수로 사용됩니다.
EPISODES = 50000

# selfplay에서만 쓰이는 에이전트 수입니다.
AGENTS = 4

# 기존 가중치를 지우고 새로 시작할지 여부입니다.
RESET = True

# 외부 기본 가중치를 먼저 받아오고 싶으면 True로 두세요.
DOWNLOAD_BASE = False

# .json.gz 외에 평문 .json도 같이 저장할지 여부입니다.
SAVE_JSON = False

# 자동 저장 주기(ms)
AUTOSAVE_MS = 60000

# 진행 로그 출력 주기(ms)
PROGRESS_MS = 30000

# 기록 정책을 바꾸고 싶을 때만 숫자를 넣으세요.
# 기본값 유지하려면 None
RECORDING_MIN_SCORE = None
SELFPLAY_MIN_REWARD = None

print("설정 완료")
print(f"- MODE={MODE}")
print(f"- DIFFICULTY={DIFFICULTY}")
print(f"- EPISODES={EPISODES}")
print(f"- AGENTS={AGENTS}")
print(f"- OUTPUT_DIR={OUTPUT_DIR or '(미사용)'}")


In [ ]:
# ==============================
# 2) Google Drive 마운트 (선택)
# ==============================

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive 마운트 완료')
else:
    print('Drive 마운트를 건너뜁니다.')


In [ ]:
# ==============================
# 3) Node.js / npm 준비
# ==============================

import shutil
import subprocess

def run(cmd, cwd=None, shell=False):
    if shell:
        print(f"$ {cmd}")
        subprocess.run(cmd, cwd=cwd, shell=True, check=True)
    else:
        print("$", " ".join(cmd))
        subprocess.run(cmd, cwd=cwd, check=True)

node_path = shutil.which('node')
npm_path = shutil.which('npm')

if node_path and npm_path:
    print('이미 Node.js / npm 이 설치되어 있습니다.')
else:
    print('Node.js / npm 이 없어서 설치를 진행합니다.')
    run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -", shell=True)
    run(["apt-get", "install", "-y", "nodejs"])

run(["node", "-v"])
run(["npm", "-v"])


In [ ]:
# ==============================
# 4) 프로젝트 준비 및 의존성 설치
# ==============================

import os

# REPO_URL을 입력한 경우 clone 또는 pull 합니다.
if REPO_URL.strip():
    if os.path.isdir(PROJECT_DIR):
        print('프로젝트 폴더가 이미 있어서 git pull 을 시도합니다.')
        run(["git", "-C", PROJECT_DIR, "pull"])
    else:
        run(["git", "clone", REPO_URL, PROJECT_DIR])
else:
    print('REPO_URL이 비어 있습니다.')
    print('이미 프로젝트를 업로드해둔 경우 PROJECT_DIR 경로만 맞으면 됩니다.')

if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"프로젝트 폴더를 찾지 못했습니다: {PROJECT_DIR}\n"
        "REPO_URL을 입력하거나, 왼쪽 파일 탭/Drive를 통해 프로젝트를 먼저 올려주세요."
    )

run(["npm", "install"], cwd=PROJECT_DIR)
run(["node", "scripts/colab-train.js", "--help"], cwd=PROJECT_DIR)
print('프로젝트 준비 완료')


## 학습 실행

아래 셀은 위 설정값을 바탕으로 실제 학습을 시작합니다.

- `MODE="solo"`이면 솔로 시뮬레이터 학습
- `MODE="selfplay"`이면 에이전트끼리 붙는 self-play 학습
- 로그는 셀 출력으로 계속 흘러나옵니다.


In [ ]:
# ==============================
# 5) 학습 실행
# ==============================

import shlex
import subprocess

cmd = [
    "node",
    "scripts/colab-train.js",
    "--mode", MODE,
    "--difficulty", DIFFICULTY,
    "--episodes", str(EPISODES),
    "--autosave-ms", str(AUTOSAVE_MS),
    "--progress-ms", str(PROGRESS_MS),
]

if MODE == "selfplay":
    cmd += ["--agents", str(AGENTS)]

if RESET:
    cmd.append("--reset")

if DOWNLOAD_BASE:
    cmd.append("--download-base")

if SAVE_JSON:
    cmd.append("--save-json")

if OUTPUT_DIR:
    cmd += ["--output-dir", OUTPUT_DIR]

if RECORDING_MIN_SCORE is not None:
    cmd += ["--recording-min-score", str(RECORDING_MIN_SCORE)]

if SELFPLAY_MIN_REWARD is not None:
    cmd += ["--selfplay-min-reward", str(SELFPLAY_MIN_REWARD)]

print('실행 명령:')
print(shlex.join(cmd))

# 로그를 실시간으로 보기 위해 Popen을 사용합니다.
process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'학습 실행 실패: 종료 코드 {return_code}')

print('학습 완료')


In [ ]:
# ==============================
# 6) 결과 파일 확인
# ==============================

from pathlib import Path

def print_file_summary(file_path):
    size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"- {file_path.name} | {size_mb:.2f} MB")

project_path = Path(PROJECT_DIR)
weight_files = sorted(project_path.glob('ai-weights-*'))

if not weight_files:
    print('프로젝트 폴더 안에 가중치 파일이 없습니다.')
else:
    print('프로젝트 폴더의 가중치 파일:')
    for wf in weight_files:
        print_file_summary(wf)

if OUTPUT_DIR:
    out_path = Path(OUTPUT_DIR)
    if out_path.exists():
        print('\n출력 폴더의 가중치 파일:')
        copied = sorted(out_path.glob('ai-weights-*'))
        for wf in copied:
            print_file_summary(wf)
    else:
        print('\n출력 폴더가 아직 없습니다:', OUTPUT_DIR)


In [ ]:
# ==============================
# 7) 결과 파일 바로 다운로드 (선택)
# ==============================

# 이 셀은 코랩 브라우저로 바로 파일을 다운로드하고 싶을 때만 실행하세요.
from google.colab import files
import os

target_path = os.path.join(PROJECT_DIR, f'ai-weights-{DIFFICULTY}.json.gz')
if not os.path.exists(target_path):
    raise FileNotFoundError(f'다운로드할 파일이 없습니다: {target_path}')

print('다운로드 대상:', target_path)
files.download(target_path)
